In [1]:
# ============================================================
# PHARMAOPT — NOTEBOOK 06
# PHARMACEUTICAL ANALYTICS
# ============================================================

import pandas as pd
import numpy as np
import mysql.connector
import getpass

print("PharmaOpt — Pharmaceutical Analytics")
print("=" * 55)

mysql_password = getpass.getpass("Enter your MySQL password: ")

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=mysql_password,
    database="pharmaopt"
)

print("Connected to PharmaOpt:", connection.is_connected())

PharmaOpt — Pharmaceutical Analytics


Enter your MySQL password:  ········


Connected to PharmaOpt: True


In [2]:
# ============================================================
# SQL QUERY HELPER
# ============================================================

def query_df(sql, params=None):
    """
    Run a SELECT query against PharmaOpt
    and return the result as a pandas DataFrame.
    """

    cursor = connection.cursor()

    cursor.execute(sql, params or ())

    rows = cursor.fetchall()

    columns = [
        description[0]
        for description in cursor.description
    ]

    cursor.close()

    return pd.DataFrame(
        rows,
        columns=columns
    )

print("SQL → pandas helper ready.")

SQL → pandas helper ready.


In [4]:
# ============================================================
# DATABASE HEALTH CHECK
# ============================================================

database_summary = query_df("""
SELECT 'drug_product' AS table_name, COUNT(*) AS row_count
FROM drug_product

UNION ALL

SELECT 'drug_package' AS table_name, COUNT(*) AS row_count
FROM drug_package

UNION ALL

SELECT 'shortage_event' AS table_name, COUNT(*) AS row_count
FROM shortage_event

UNION ALL

SELECT 'recall_event' AS table_name, COUNT(*) AS row_count
FROM recall_event

UNION ALL

SELECT 'recall_package' AS table_name, COUNT(*) AS row_count
FROM recall_package;
""")

database_summary

,table_name,row_count
0,drug_product,138217
1,drug_package,256383
2,shortage_event,1601
3,recall_event,17975
4,recall_package,25577


In [5]:
# ============================================================
# ANALYSIS 1 — PRIOR RECALL HISTORY BEFORE FIRST SHORTAGE
# ============================================================

supply_signal_df = query_df("""
WITH first_shortage AS (

    SELECT
        linked_package_ndc AS package_ndc,
        MIN(initial_posting_date) AS first_shortage_date,
        MAX(initial_posting_date) AS latest_shortage_date,
        COUNT(*) AS shortage_event_count

    FROM shortage_event

    WHERE linked_package_ndc IS NOT NULL
      AND initial_posting_date IS NOT NULL

    GROUP BY linked_package_ndc
),

prior_recall AS (

    SELECT
        fs.package_ndc,

        COUNT(
            DISTINCT re.recall_number
        ) AS prior_recall_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class I'
                THEN re.recall_number
            END
        ) AS prior_class_I_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class II'
                THEN re.recall_number
            END
        ) AS prior_class_II_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class III'
                THEN re.recall_number
            END
        ) AS prior_class_III_count,

        MAX(
            re.recall_initiation_date
        ) AS most_recent_prior_recall

    FROM first_shortage fs

    LEFT JOIN recall_package rp
        ON fs.package_ndc = rp.linked_package_ndc

    LEFT JOIN recall_event re
        ON rp.recall_number = re.recall_number
       AND re.recall_initiation_date < fs.first_shortage_date

    GROUP BY fs.package_ndc
)

SELECT
    fs.package_ndc,

    dp.product_ndc,
    dp.generic_name,
    dp.brand_name,
    dp.labeler_name,
    dp.dosage_form,
    dp.marketing_category,

    fs.first_shortage_date,
    fs.latest_shortage_date,
    fs.shortage_event_count,

    pr.prior_recall_count,
    pr.prior_class_I_count,
    pr.prior_class_II_count,
    pr.prior_class_III_count,
    pr.most_recent_prior_recall,

    DATEDIFF(
        fs.first_shortage_date,
        pr.most_recent_prior_recall
    ) AS days_recall_to_shortage

FROM first_shortage fs

INNER JOIN drug_package pkg
    ON fs.package_ndc = pkg.package_ndc

INNER JOIN drug_product dp
    ON pkg.product_id = dp.product_id

LEFT JOIN prior_recall pr
    ON fs.package_ndc = pr.package_ndc

ORDER BY
    pr.prior_recall_count DESC,
    fs.first_shortage_date;
""")

print(
    "Supply-risk analytical records:",
    f"{len(supply_signal_df):,}"
)

supply_signal_df.head(20)

Supply-risk analytical records: 1,393


,package_ndc,product_ndc,generic_name,brand_name,labeler_name,dosage_form,marketing_category,first_shortage_date,latest_shortage_date,shortage_event_count,prior_recall_count,prior_class_I_count,prior_class_II_count,prior_class_III_count,most_recent_prior_recall,days_recall_to_shortage
0,0941-0409-07,0941-0409,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL Low Calcium with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
1,0941-0411-06,0941-0411,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
2,0941-0411-07,0941-0411,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
3,0941-0413-06,0941-0413,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
4,0941-0413-07,0941-0413,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
5,0941-0415-06,0941-0415,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
6,0941-0415-07,0941-0415,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
7,0941-0424-51,0941-0424,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL Low Calcium with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
8,0941-0426-52,0941-0426,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0
9,0941-0426-53,0941-0426,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,9,2,7,0,2024-06-17,777.0


In [6]:
# ============================================================
# PRIOR RECALL SIGNAL SUMMARY
# ============================================================

total_shortage_packages = len(supply_signal_df)

packages_with_prior_recall = (
    supply_signal_df["prior_recall_count"] > 0
).sum()

packages_without_prior_recall = (
    supply_signal_df["prior_recall_count"] == 0
).sum()

prior_recall_rate = (
    packages_with_prior_recall
    / total_shortage_packages
    * 100
)

prior_recall_cases = supply_signal_df[
    supply_signal_df["prior_recall_count"] > 0
].copy()

print("PHARMAOPT — PRIOR RECALL SIGNAL")
print("=" * 60)

print(
    "Shortage packages analyzed:",
    f"{total_shortage_packages:,}"
)

print(
    "Had ≥1 recall BEFORE shortage:",
    f"{packages_with_prior_recall:,}"
)

print(
    "Had no prior recall:",
    f"{packages_without_prior_recall:,}"
)

print(
    "Prior-recall rate:",
    f"{prior_recall_rate:.2f}%"
)

print()

if len(prior_recall_cases) > 0:

    print(
        "Median days from most recent prior recall → shortage:",
        f"{prior_recall_cases['days_recall_to_shortage'].median():,.0f}"
    )

    print(
        "Average prior recalls among affected packages:",
        f"{prior_recall_cases['prior_recall_count'].mean():.2f}"
    )

    print()

    print("PRIOR RECALL CLASSIFICATION COUNTS")
    print("-" * 60)

    print(
        "Class I:",
        int(prior_recall_cases["prior_class_I_count"].sum())
    )

    print(
        "Class II:",
        int(prior_recall_cases["prior_class_II_count"].sum())
    )

    print(
        "Class III:",
        int(prior_recall_cases["prior_class_III_count"].sum())
    )

PHARMAOPT — PRIOR RECALL SIGNAL
Shortage packages analyzed: 1,393
Had ≥1 recall BEFORE shortage: 245
Had no prior recall: 1,148
Prior-recall rate: 17.59%

Median days from most recent prior recall → shortage: 1,267
Average prior recalls among affected packages: 3.26

PRIOR RECALL CLASSIFICATION COUNTS
------------------------------------------------------------
Class I: 177
Class II: 522
Class III: 99


In [7]:
# ============================================================
# ANALYSIS 2 — RECALL RECENCY BEFORE SHORTAGE
# ============================================================

recall_window_df = query_df("""
WITH first_shortage AS (

    SELECT
        linked_package_ndc AS package_ndc,
        MIN(initial_posting_date) AS first_shortage_date

    FROM shortage_event

    WHERE linked_package_ndc IS NOT NULL
      AND initial_posting_date IS NOT NULL

    GROUP BY linked_package_ndc
),

recall_history AS (

    SELECT
        fs.package_ndc,
        fs.first_shortage_date,
        re.recall_number,
        re.classification,
        re.recall_initiation_date,

        DATEDIFF(
            fs.first_shortage_date,
            re.recall_initiation_date
        ) AS days_before_shortage

    FROM first_shortage fs

    INNER JOIN recall_package rp
        ON fs.package_ndc = rp.linked_package_ndc

    INNER JOIN recall_event re
        ON rp.recall_number = re.recall_number

    WHERE re.recall_initiation_date < fs.first_shortage_date
)

SELECT
    package_ndc,

    COUNT(DISTINCT recall_number) AS lifetime_prior_recalls,

    COUNT(DISTINCT CASE
        WHEN days_before_shortage <= 90
        THEN recall_number
    END) AS recalls_90d,

    COUNT(DISTINCT CASE
        WHEN days_before_shortage <= 180
        THEN recall_number
    END) AS recalls_180d,

    COUNT(DISTINCT CASE
        WHEN days_before_shortage <= 365
        THEN recall_number
    END) AS recalls_1y,

    COUNT(DISTINCT CASE
        WHEN days_before_shortage <= 730
        THEN recall_number
    END) AS recalls_2y,

    COUNT(DISTINCT CASE
        WHEN days_before_shortage <= 1825
        THEN recall_number
    END) AS recalls_5y

FROM recall_history

GROUP BY package_ndc;
""")

print("Packages with any prior recall:", len(recall_window_df))
recall_window_df.head(10)

Packages with any prior recall: 245


,package_ndc,lifetime_prior_recalls,recalls_90d,recalls_180d,recalls_1y,recalls_2y,recalls_5y
0,0002-8213-05,2,0,0,0,1,2
1,0006-3066-03,1,0,1,1,1,1
2,0009-0775-26,4,0,0,0,0,3
3,0009-0870-26,4,0,0,0,0,3
4,0009-0902-18,4,0,0,0,0,3
5,0009-3073-01,1,0,0,0,0,0
6,0009-3073-03,1,0,0,0,0,0
7,0009-3073-22,1,0,0,0,0,0
8,0009-3073-23,1,0,0,0,0,0
9,0009-3475-01,1,0,0,0,0,0


In [8]:
# ============================================================
# RECALL WINDOW SUMMARY
# ============================================================

total_shortage_packages = len(supply_signal_df)

windows = {
    "90 days": "recalls_90d",
    "180 days": "recalls_180d",
    "1 year": "recalls_1y",
    "2 years": "recalls_2y",
    "5 years": "recalls_5y",
    "Any prior history": "lifetime_prior_recalls"
}

print("RECALL HISTORY BEFORE FIRST SHORTAGE")
print("=" * 70)

for label, column in windows.items():

    packages = (
        recall_window_df[column] > 0
    ).sum()

    rate = (
        packages / total_shortage_packages * 100
    )

    print(
        f"{label:<20}"
        f"{packages:>6,} packages"
        f"{rate:>12.2f}%"
    )

RECALL HISTORY BEFORE FIRST SHORTAGE
90 days                 15 packages        1.08%
180 days                27 packages        1.94%
1 year                  38 packages        2.73%
2 years                 76 packages        5.46%
5 years                154 packages       11.06%
Any prior history      245 packages       17.59%


In [9]:
# ============================================================
# BUILD SHORTAGE-POSITIVE FEATURE TABLE
# ============================================================

shortage_features = supply_signal_df.merge(
    recall_window_df[
        [
            "package_ndc",
            "recalls_90d",
            "recalls_180d",
            "recalls_1y",
            "recalls_2y",
            "recalls_5y"
        ]
    ],
    on="package_ndc",
    how="left"
)

recall_feature_columns = [
    "recalls_90d",
    "recalls_180d",
    "recalls_1y",
    "recalls_2y",
    "recalls_5y"
]

shortage_features[recall_feature_columns] = (
    shortage_features[recall_feature_columns]
    .fillna(0)
    .astype(int)
)

# Every row in this table represents a package
# that experienced a shortage.
shortage_features["shortage_target"] = 1

print("SHORTAGE-POSITIVE FEATURE TABLE")
print("=" * 60)
print("Rows:", f"{len(shortage_features):,}")
print("Columns:", len(shortage_features.columns))
print()

shortage_features.head(10)

SHORTAGE-POSITIVE FEATURE TABLE
Rows: 1,393
Columns: 22



,package_ndc,product_ndc,generic_name,brand_name,labeler_name,dosage_form,marketing_category,first_shortage_date,latest_shortage_date,shortage_event_count,...,prior_class_II_count,prior_class_III_count,most_recent_prior_recall,days_recall_to_shortage,recalls_90d,recalls_180d,recalls_1y,recalls_2y,recalls_5y,shortage_target
0,0941-0409-07,0941-0409,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL Low Calcium with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
1,0941-0411-06,0941-0411,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
2,0941-0411-07,0941-0411,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
3,0941-0413-06,0941-0413,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
4,0941-0413-07,0941-0413,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
5,0941-0415-06,0941-0415,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
6,0941-0415-07,0941-0415,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
7,0941-0424-51,0941-0424,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL Low Calcium with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
8,0941-0426-52,0941-0426,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1
9,0941-0426-53,0941-0426,"sodium chloride, sodium lactate, calcium chlor...",DIANEAL PD-2 with Dextrose,Vantive US Healthcare LLC,"INJECTION, SOLUTION",NDA,2026-08-03,2026-08-03,1,...,7,0,2024-06-17,777.0,0,0,0,0,5,1


In [10]:
# ============================================================
# ANALYSIS 3 — SHORTAGE TIME COVERAGE
# ============================================================

shortage_timeline = query_df("""
SELECT
    YEAR(initial_posting_date) AS shortage_year,
    COUNT(*) AS shortage_events,
    COUNT(DISTINCT linked_package_ndc) AS unique_linked_packages
FROM shortage_event
WHERE initial_posting_date IS NOT NULL
GROUP BY YEAR(initial_posting_date)
ORDER BY shortage_year;
""")

shortage_range = query_df("""
SELECT
    MIN(initial_posting_date) AS earliest_shortage,
    MAX(initial_posting_date) AS latest_shortage,
    COUNT(*) AS dated_events
FROM shortage_event
WHERE initial_posting_date IS NOT NULL;
""")

print("SHORTAGE DATA COVERAGE")
print("=" * 60)

print("Earliest shortage:",
      shortage_range.loc[0, "earliest_shortage"])

print("Latest shortage:",
      shortage_range.loc[0, "latest_shortage"])

print("Dated shortage events:",
      f"{shortage_range.loc[0, 'dated_events']:,}")

print()
print("EVENTS BY YEAR")
print("-" * 60)

shortage_timeline

SHORTAGE DATA COVERAGE
Earliest shortage: 2012-01-01
Latest shortage: 2026-09-21
Dated shortage events: 1,601

EVENTS BY YEAR
------------------------------------------------------------


,shortage_year,shortage_events,unique_linked_packages
0,2012,76,74
1,2015,6,6
2,2016,27,27
3,2017,100,95
4,2018,182,168
5,2019,20,18
6,2020,145,136
7,2021,53,50
8,2022,171,160
9,2023,345,317


In [11]:
# ============================================================
# LOAD NDC PRODUCT ACTIVITY DATES
# ============================================================

import zipfile
import json

ndc_zip = "drug-ndc-0001-of-0001.json.zip"

with zipfile.ZipFile(ndc_zip, "r") as z:
    json_filename = z.namelist()[0]

    with z.open(json_filename) as f:
        ndc_raw = json.load(f)

ndc_activity_raw = pd.json_normalize(
    ndc_raw["results"]
)

# Show fields related to time / marketing activity
activity_columns = [
    column
    for column in ndc_activity_raw.columns
    if (
        "date" in column.lower()
        or "marketing" in column.lower()
        or "expiration" in column.lower()
    )
]

print("NDC ACTIVITY-RELATED FIELDS")
print("=" * 60)

for column in activity_columns:
    print(column)

NDC ACTIVITY-RELATED FIELDS
listing_expiration_date
marketing_category
marketing_start_date
marketing_end_date


In [12]:
# ============================================================
# ANALYSIS 4 — NDC PRODUCT ACTIVITY DATE AUDIT
# ============================================================

activity_df = ndc_activity_raw[
    [
        "product_id",
        "product_ndc",
        "marketing_start_date",
        "marketing_end_date",
        "listing_expiration_date"
    ]
].copy()


# ------------------------------------------------------------
# CONVERT FDA YYYYMMDD DATES
# ------------------------------------------------------------

date_columns = [
    "marketing_start_date",
    "marketing_end_date",
    "listing_expiration_date"
]

for column in date_columns:
    activity_df[column] = pd.to_datetime(
        activity_df[column].astype("string"),
        format="%Y%m%d",
        errors="coerce"
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("NDC ACTIVITY DATE AUDIT")
print("=" * 65)

print("Total products:", f"{len(activity_df):,}")

for column in date_columns:

    valid = activity_df[column].notna().sum()
    missing = activity_df[column].isna().sum()

    print()
    print(column)
    print("-" * 40)
    print("Valid dates:", f"{valid:,}")
    print("Missing dates:", f"{missing:,}")

    if valid > 0:
        print(
            "Earliest:",
            activity_df[column].min().date()
        )
        print(
            "Latest:",
            activity_df[column].max().date()
        )

NDC ACTIVITY DATE AUDIT
Total products: 138,217

marketing_start_date
----------------------------------------
Valid dates: 138,217
Missing dates: 0
Earliest: 1900-01-01
Latest: 2026-09-24

marketing_end_date
----------------------------------------
Valid dates: 3,879
Missing dates: 134,338
Earliest: 2026-09-26
Latest: 2049-12-31

listing_expiration_date
----------------------------------------
Valid dates: 134,331
Missing dates: 3,886
Earliest: 2017-12-31
Latest: 2027-12-31


In [13]:
# ============================================================
# MARKETING START YEAR DISTRIBUTION
# ============================================================

activity_df["marketing_start_year"] = (
    activity_df["marketing_start_date"].dt.year
)

marketing_year_summary = (
    activity_df
    .dropna(subset=["marketing_start_year"])
    .groupby("marketing_start_year")
    .size()
    .reset_index(name="product_count")
    .sort_values("marketing_start_year")
)

print("MARKETING START YEAR COVERAGE")
print("=" * 60)

marketing_year_summary.tail(20)

MARKETING START YEAR COVERAGE


,marketing_start_year,product_count
67,2007,1665
68,2008,1407
69,2009,2268
70,2010,2472
71,2011,2801
72,2012,2979
73,2013,3038
74,2014,3677
75,2015,4786
76,2016,4247


In [14]:
# ============================================================
# MODELING COHORT 1
# ACTIVE PACKAGES AT 2025-01-01
# ============================================================

index_date = pd.Timestamp("2025-01-01")
horizon_end = pd.Timestamp("2025-12-31")

print("MODEL COHORT")
print("=" * 60)
print("Index date:", index_date.date())
print("Prediction horizon end:", horizon_end.date())


# ------------------------------------------------------------
# GET PACKAGE + PRODUCT INFORMATION FROM MYSQL
# ------------------------------------------------------------

package_population = query_df("""
SELECT
    pkg.package_ndc,
    pkg.product_id,
    dp.product_ndc,
    dp.generic_name,
    dp.brand_name,
    dp.labeler_name,
    dp.dosage_form,
    dp.marketing_category
FROM drug_package pkg
INNER JOIN drug_product dp
    ON pkg.product_id = dp.product_id;
""")


# ------------------------------------------------------------
# MERGE FDA MARKETING ACTIVITY DATES
# ------------------------------------------------------------

cohort_2025 = package_population.merge(
    activity_df[
        [
            "product_id",
            "marketing_start_date",
            "marketing_end_date",
            "listing_expiration_date"
        ]
    ],
    on="product_id",
    how="left"
)


# ------------------------------------------------------------
# DEFINE ACTIVE ON INDEX DATE
# ------------------------------------------------------------

active_mask = (
    cohort_2025["marketing_start_date"].notna()
    &
    (cohort_2025["marketing_start_date"] <= index_date)
    &
    (
        cohort_2025["marketing_end_date"].isna()
        |
        (cohort_2025["marketing_end_date"] >= index_date)
    )
)

active_2025 = (
    cohort_2025[active_mask]
    .copy()
    .reset_index(drop=True)
)

print()
print("Total package records:", f"{len(cohort_2025):,}")
print(
    "Active packages at 2025-01-01:",
    f"{len(active_2025):,}"
)

MODEL COHORT
Index date: 2025-01-01
Prediction horizon end: 2025-12-31

Total package records: 256,383
Active packages at 2025-01-01: 221,667


In [15]:
# ============================================================
# CREATE 2025 SHORTAGE TARGET
# ============================================================

shortages_2025 = query_df("""
SELECT DISTINCT
    linked_package_ndc AS package_ndc
FROM shortage_event
WHERE linked_package_ndc IS NOT NULL
  AND initial_posting_date >= '2025-01-01'
  AND initial_posting_date <= '2025-12-31';
""")

shortage_packages_2025 = set(
    shortages_2025["package_ndc"]
    .dropna()
    .astype(str)
)

active_2025["shortage_target"] = (
    active_2025["package_ndc"]
    .astype(str)
    .isin(shortage_packages_2025)
    .astype(int)
)


print("2025 TARGET DISTRIBUTION")
print("=" * 60)

print(
    "Active packages:",
    f"{len(active_2025):,}"
)

print(
    "Shortage = 1:",
    f"{active_2025['shortage_target'].sum():,}"
)

print(
    "Shortage = 0:",
    f"{(active_2025['shortage_target'] == 0).sum():,}"
)

shortage_rate = (
    active_2025["shortage_target"].mean() * 100
)

print(
    "Shortage rate:",
    f"{shortage_rate:.4f}%"
)

2025 TARGET DISTRIBUTION
Active packages: 221,667
Shortage = 1: 96
Shortage = 0: 221,571
Shortage rate: 0.0433%


In [16]:
# ============================================================
# PRE-2025 RECALL FEATURES
# ============================================================

recall_features_2025 = query_df("""
SELECT
    rp.linked_package_ndc AS package_ndc,

    COUNT(
        DISTINCT re.recall_number
    ) AS prior_recall_count,

    COUNT(
        DISTINCT CASE
            WHEN re.classification = 'Class I'
            THEN re.recall_number
        END
    ) AS prior_class_I_count,

    COUNT(
        DISTINCT CASE
            WHEN re.classification = 'Class II'
            THEN re.recall_number
        END
    ) AS prior_class_II_count,

    COUNT(
        DISTINCT CASE
            WHEN re.classification = 'Class III'
            THEN re.recall_number
        END
    ) AS prior_class_III_count,

    COUNT(
        DISTINCT CASE
            WHEN re.recall_initiation_date >= '2024-01-01'
             AND re.recall_initiation_date < '2025-01-01'
            THEN re.recall_number
        END
    ) AS recalls_prior_1y,

    COUNT(
        DISTINCT CASE
            WHEN re.recall_initiation_date >= '2023-01-01'
             AND re.recall_initiation_date < '2025-01-01'
            THEN re.recall_number
        END
    ) AS recalls_prior_2y,

    COUNT(
        DISTINCT CASE
            WHEN re.recall_initiation_date >= '2020-01-01'
             AND re.recall_initiation_date < '2025-01-01'
            THEN re.recall_number
        END
    ) AS recalls_prior_5y,

    MAX(
        re.recall_initiation_date
    ) AS most_recent_prior_recall

FROM recall_package rp

INNER JOIN recall_event re
    ON rp.recall_number = re.recall_number

WHERE rp.linked_package_ndc IS NOT NULL
  AND re.recall_initiation_date < '2025-01-01'

GROUP BY rp.linked_package_ndc;
""")


model_2025 = active_2025.merge(
    recall_features_2025,
    on="package_ndc",
    how="left"
)


recall_numeric_columns = [
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y"
]

model_2025[recall_numeric_columns] = (
    model_2025[recall_numeric_columns]
    .fillna(0)
    .astype(int)
)


model_2025["most_recent_prior_recall"] = pd.to_datetime(
    model_2025["most_recent_prior_recall"]
)


model_2025["days_since_last_recall"] = (
    index_date
    -
    model_2025["most_recent_prior_recall"]
).dt.days


print("FIRST PHARMAOPT MODELING TABLE")
print("=" * 60)

print("Rows:", f"{len(model_2025):,}")
print("Columns:", len(model_2025.columns))

print()
print("Target counts:")
print(
    model_2025["shortage_target"]
    .value_counts()
    .sort_index()
)

model_2025.head()

FIRST PHARMAOPT MODELING TABLE
Rows: 221,667
Columns: 21

Target counts:
shortage_target
0    221571
1        96
Name: count, dtype: int64


,package_ndc,product_id,product_ndc,generic_name,brand_name,labeler_name,dosage_form,marketing_category,marketing_start_date,marketing_end_date,...,shortage_target,prior_recall_count,prior_class_I_count,prior_class_II_count,prior_class_III_count,recalls_prior_1y,recalls_prior_2y,recalls_prior_5y,most_recent_prior_recall,days_since_last_recall
0,0002-0013-03,0002-0013_15810e26-b479-41b8-9d13-9dc503a71a2a,0002-0013,Insulin human,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING,1983-06-27,NaT,...,0,0,0,0,0,0,0,0,NaT,NaN
1,0002-0013-10,0002-0013_15810e26-b479-41b8-9d13-9dc503a71a2a,0002-0013,Insulin human,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING,1983-06-27,NaT,...,0,0,0,0,0,0,0,0,NaT,NaN
2,0002-0095-00,0002-0095_15810e26-b479-41b8-9d13-9dc503a71a2a,0002-0095,Insulin human,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING,2015-12-29,NaT,...,0,0,0,0,0,0,0,0,NaT,NaN
3,0002-0096-00,0002-0096_15810e26-b479-41b8-9d13-9dc503a71a2a,0002-0096,Insulin human,NaN,Eli Lilly and Company,"INJECTION, SOLUTION",DRUG FOR FURTHER PROCESSING,1994-03-31,NaT,...,0,0,0,0,0,0,0,0,NaT,NaN
4,0002-0113-03,0002-0113_55aa4078-e1e8-4338-b2b5-a9298b3c16e2,0002-0113,Insulin human,NaN,Eli Lilly and Company,"INJECTION, SUSPENSION",DRUG FOR FURTHER PROCESSING,1983-06-27,NaT,...,0,0,0,0,0,0,0,0,NaT,NaN


In [17]:
# ============================================================
# ANALYSIS 5 — AUDIT 2025 POSITIVE SHORTAGE CASES
# ============================================================

positive_2025 = model_2025[
    model_2025["shortage_target"] == 1
].copy()

print("2025 POSITIVE CASE AUDIT")
print("=" * 65)

print(
    "Positive package NDCs:",
    f"{positive_2025['package_ndc'].nunique():,}"
)

print(
    "Unique product IDs:",
    f"{positive_2025['product_id'].nunique():,}"
)

print(
    "Unique product NDCs:",
    f"{positive_2025['product_ndc'].nunique():,}"
)

print(
    "Unique generic names:",
    f"{positive_2025['generic_name'].nunique():,}"
)

print(
    "Unique labelers:",
    f"{positive_2025['labeler_name'].nunique():,}"
)

print()


# ------------------------------------------------------------
# TOP PRODUCT FAMILIES AMONG POSITIVE CASES
# ------------------------------------------------------------

positive_product_counts = (
    positive_2025
    .groupby(
        [
            "product_ndc",
            "generic_name",
            "labeler_name"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="positive_packages")
    .sort_values(
        "positive_packages",
        ascending=False
    )
)

print("TOP POSITIVE PRODUCT FAMILIES")
print("-" * 65)

positive_product_counts.head(20)

2025 POSITIVE CASE AUDIT
Positive package NDCs: 96
Unique product IDs: 83
Unique product NDCs: 83
Unique generic names: 41
Unique labelers: 25

TOP POSITIVE PRODUCT FAMILIES
-----------------------------------------------------------------


,product_ndc,generic_name,labeler_name,positive_packages
53,0781-2868,Omeprazole,Sandoz Inc,4
51,0781-2234,Omeprazole,Sandoz Inc,3
52,0781-2859,Omeprazole,Sandoz Inc,3
41,0591-0347,Hydrochlorothiazide,"Actavis Pharma, Inc.",2
79,62037-720,Potassium Chloride,"Actavis Pharma, Inc.",2
36,0409-7101,Sodium Chloride,"Hospira, Inc.",2
74,60951-602,Oxycodone and Acetaminophen,"Par Health USA, LLC",2
6,0002-8235,Insulin lispro-aabc,Eli Lilly and Company,2
1,0002-4491,lasmiditan,Eli Lilly and Company,2
2,0002-7623,Pemetrexed disodium,Eli Lilly and Company,1


In [18]:
# ============================================================
# ANALYSIS 6 — SHORTAGE ONSETS BY YEAR
# ============================================================

annual_shortages = query_df("""
SELECT
    YEAR(initial_posting_date) AS shortage_year,

    COUNT(*) AS shortage_rows,

    COUNT(
        DISTINCT linked_package_ndc
    ) AS unique_packages,

    COUNT(
        DISTINCT source_package_ndc
    ) AS unique_source_packages

FROM shortage_event

WHERE initial_posting_date IS NOT NULL

GROUP BY
    YEAR(initial_posting_date)

ORDER BY
    shortage_year;
""")

print("HISTORICAL SHORTAGE ONSETS")
print("=" * 65)

annual_shortages

HISTORICAL SHORTAGE ONSETS


,shortage_year,shortage_rows,unique_packages,unique_source_packages
0,2012,76,74,76
1,2015,6,6,6
2,2016,27,27,27
3,2017,100,95,100
4,2018,182,168,177
5,2019,20,18,20
6,2020,145,136,144
7,2021,53,50,53
8,2022,171,160,171
9,2023,345,317,343


In [19]:
# ============================================================
# PHARMAOPT — MULTI-YEAR MODELING DATASET
# 2022 → 2025
# ============================================================

model_years = [2022, 2023, 2024, 2025]

modeling_frames = []

print("BUILDING PHARMAOPT MODELING PANEL")
print("=" * 70)


for year in model_years:

    index_date = pd.Timestamp(f"{year}-01-01")
    horizon_end = pd.Timestamp(f"{year}-12-31")

    print(f"\nProcessing {year}...")


    # --------------------------------------------------------
    # 1. ACTIVE PACKAGE POPULATION
    # --------------------------------------------------------

    year_population = package_population.merge(
        activity_df[
            [
                "product_id",
                "marketing_start_date",
                "marketing_end_date",
                "listing_expiration_date"
            ]
        ],
        on="product_id",
        how="left"
    )


    active_mask = (
        year_population["marketing_start_date"].notna()
        &
        (
            year_population["marketing_start_date"]
            <= index_date
        )
        &
        (
            year_population["marketing_end_date"].isna()
            |
            (
                year_population["marketing_end_date"]
                >= index_date
            )
        )
    )


    year_df = (
        year_population[active_mask]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 2. SHORTAGE TARGET
    # --------------------------------------------------------

    shortage_query = """
    SELECT DISTINCT
        linked_package_ndc AS package_ndc

    FROM shortage_event

    WHERE linked_package_ndc IS NOT NULL
      AND initial_posting_date >= %s
      AND initial_posting_date < %s;
    """

    next_year_date = pd.Timestamp(
        f"{year + 1}-01-01"
    )

    year_shortages = query_df(
        shortage_query,
        (
            index_date.date(),
            next_year_date.date()
        )
    )


    shortage_set = set(
        year_shortages["package_ndc"]
        .dropna()
        .astype(str)
    )


    year_df["shortage_target"] = (
        year_df["package_ndc"]
        .astype(str)
        .isin(shortage_set)
        .astype(int)
    )


    # --------------------------------------------------------
    # 3. RECALL FEATURES KNOWN BEFORE INDEX DATE
    # --------------------------------------------------------

    recall_query = """
    SELECT

        rp.linked_package_ndc AS package_ndc,

        COUNT(
            DISTINCT re.recall_number
        ) AS prior_recall_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class I'
                THEN re.recall_number
            END
        ) AS prior_class_I_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class II'
                THEN re.recall_number
            END
        ) AS prior_class_II_count,

        COUNT(
            DISTINCT CASE
                WHEN re.classification = 'Class III'
                THEN re.recall_number
            END
        ) AS prior_class_III_count,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_1y,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_2y,

        COUNT(
            DISTINCT CASE
                WHEN re.recall_initiation_date >= %s
                THEN re.recall_number
            END
        ) AS recalls_prior_5y,

        MAX(
            re.recall_initiation_date
        ) AS most_recent_prior_recall

    FROM recall_package rp

    INNER JOIN recall_event re
        ON rp.recall_number = re.recall_number

    WHERE rp.linked_package_ndc IS NOT NULL
      AND re.recall_initiation_date < %s

    GROUP BY rp.linked_package_ndc;
    """


    date_1y = (
        index_date - pd.DateOffset(years=1)
    ).date()

    date_2y = (
        index_date - pd.DateOffset(years=2)
    ).date()

    date_5y = (
        index_date - pd.DateOffset(years=5)
    ).date()


    year_recalls = query_df(
        recall_query,
        (
            date_1y,
            date_2y,
            date_5y,
            index_date.date()
        )
    )


    # --------------------------------------------------------
    # 4. MERGE RECALL FEATURES
    # --------------------------------------------------------

    year_df = year_df.merge(
        year_recalls,
        on="package_ndc",
        how="left"
    )


    recall_columns = [
        "prior_recall_count",
        "prior_class_I_count",
        "prior_class_II_count",
        "prior_class_III_count",
        "recalls_prior_1y",
        "recalls_prior_2y",
        "recalls_prior_5y"
    ]


    year_df[recall_columns] = (
        year_df[recall_columns]
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # 5. PRODUCT AGE
    # --------------------------------------------------------

    year_df["product_age_days"] = (
        index_date
        -
        year_df["marketing_start_date"]
    ).dt.days


    # --------------------------------------------------------
    # 6. RECALL RECENCY
    # --------------------------------------------------------

    year_df["most_recent_prior_recall"] = (
        pd.to_datetime(
            year_df["most_recent_prior_recall"]
        )
    )


    year_df["days_since_last_recall"] = (
        index_date
        -
        year_df["most_recent_prior_recall"]
    ).dt.days


    # Binary indicator is useful when recency is missing
    year_df["has_prior_recall"] = (
        year_df["prior_recall_count"] > 0
    ).astype(int)


    # --------------------------------------------------------
    # 7. IDENTIFY PREDICTION YEAR
    # --------------------------------------------------------

    year_df["prediction_year"] = year


    # --------------------------------------------------------
    # 8. STORE YEAR
    # --------------------------------------------------------

    modeling_frames.append(year_df)


    print(
        f"Active packages: {len(year_df):,}"
    )

    print(
        f"Shortage positives: "
        f"{year_df['shortage_target'].sum():,}"
    )


# ============================================================
# COMBINE ALL YEARS
# ============================================================

pharmaopt_model_df = pd.concat(
    modeling_frames,
    ignore_index=True
)


print("\n" + "=" * 70)
print("MULTI-YEAR MODELING DATASET COMPLETE")
print("=" * 70)

print(
    "Total package-year observations:",
    f"{len(pharmaopt_model_df):,}"
)

print(
    "Total shortage-positive observations:",
    f"{pharmaopt_model_df['shortage_target'].sum():,}"
)

print(
    "Total non-shortage observations:",
    f"{(pharmaopt_model_df['shortage_target'] == 0).sum():,}"
)

BUILDING PHARMAOPT MODELING PANEL

Processing 2022...
Active packages: 157,804
Shortage positives: 128

Processing 2023...
Active packages: 179,808
Shortage positives: 195

Processing 2024...
Active packages: 203,628
Shortage positives: 7

Processing 2025...
Active packages: 221,667
Shortage positives: 96

MULTI-YEAR MODELING DATASET COMPLETE
Total package-year observations: 762,907
Total shortage-positive observations: 426
Total non-shortage observations: 762,481


In [20]:
# ============================================================
# PHARMAOPT MODELING PANEL SUMMARY
# ============================================================

panel_summary = (
    pharmaopt_model_df
    .groupby("prediction_year")
    .agg(
        observations=(
            "package_ndc",
            "size"
        ),
        shortage_positives=(
            "shortage_target",
            "sum"
        )
    )
    .reset_index()
)

panel_summary["shortage_negatives"] = (
    panel_summary["observations"]
    -
    panel_summary["shortage_positives"]
)

panel_summary["shortage_rate_pct"] = (
    panel_summary["shortage_positives"]
    /
    panel_summary["observations"]
    * 100
)


print("PHARMAOPT MODELING PANEL")
print("=" * 75)

panel_summary

PHARMAOPT MODELING PANEL


,prediction_year,observations,shortage_positives,shortage_negatives,shortage_rate_pct
0,2022,157804,128,157676,0.081113
1,2023,179808,195,179613,0.108449
2,2024,203628,7,203621,0.003438
3,2025,221667,96,221571,0.043308


In [22]:
    # ============================================================
# INSTALL SCIKIT-LEARN INTO PHARMAOPT ENVIRONMENT
# ============================================================

import sys
import subprocess

subprocess.check_call([
    r"C:\Users\jpiza\anaconda3\Scripts\conda.exe",
    "install",
    "-n",
    "pharmaopt",
    "scikit-learn",
    "-y"
])

print("scikit-learn installed.")

scikit-learn installed.


In [25]:
# ============================================================
# PHARMAOPT v0.1 — BASELINE SHORTAGE RISK MODEL
# TRAIN: 2022–2024
# TEST:  2025
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

numeric_features = [
    "product_age_days",
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",
    "days_since_last_recall",
    "has_prior_recall"
]

categorical_features = [
    "dosage_form",
    "marketing_category"
]

features = (
    numeric_features
    + categorical_features
)

target = "shortage_target"


# ------------------------------------------------------------
# TIME-BASED TRAIN / TEST SPLIT
# ------------------------------------------------------------

train_df = pharmaopt_model_df[
    pharmaopt_model_df["prediction_year"] < 2025
].copy()

test_df = pharmaopt_model_df[
    pharmaopt_model_df["prediction_year"] == 2025
].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]


print("TRAINING SET")
print("=" * 55)
print("Rows:", f"{len(X_train):,}")
print("Shortages:", f"{y_train.sum():,}")

print()

print("2025 TEST SET")
print("=" * 55)
print("Rows:", f"{len(X_test):,}")
print("Shortages:", f"{y_test.sum():,}")


# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# BASELINE MODEL
# ------------------------------------------------------------

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear"
)

pharmaopt_v01 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ]
)


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("\nTraining PharmaOpt v0.1...")

pharmaopt_v01.fit(
    X_train,
    y_train
)

print("Training complete.")


# ------------------------------------------------------------
# GENERATE 2025 RISK SCORES
# ------------------------------------------------------------

risk_scores = pharmaopt_v01.predict_proba(
    X_test
)[:, 1]

results_2025 = test_df[
    [
        "package_ndc",
        "product_ndc",
        "generic_name",
        "brand_name",
        "labeler_name",
        "shortage_target"
    ]
].copy()

results_2025["risk_score"] = risk_scores


# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

pr_auc = average_precision_score(
    y_test,
    risk_scores
)

roc_auc = roc_auc_score(
    y_test,
    risk_scores
)

base_rate = y_test.mean()


print("\nPHARMAOPT v0.1 — 2025 RESULTS")
print("=" * 60)

print(
    "Baseline shortage rate:",
    f"{base_rate:.6f}"
)

print(
    "PR-AUC:",
    f"{pr_auc:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc:.4f}"
)

print(
    "PR-AUC lift over random:",
    f"{pr_auc / base_rate:.2f}x"
)

TRAINING SET
Rows: 541,240
Shortages: 330

2025 TEST SET
Rows: 221,667
Shortages: 96

Training PharmaOpt v0.1...
Training complete.

PHARMAOPT v0.1 — 2025 RESULTS
Baseline shortage rate: 0.000433
PR-AUC: 0.002110
ROC-AUC: 0.7440
PR-AUC lift over random: 4.87x


In [26]:
# ============================================================
# PHARMAOPT — TOP-RISK CAPTURE
# ============================================================

ranked_2025 = (
    results_2025
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

total_actual_shortages = int(
    ranked_2025["shortage_target"].sum()
)

baseline_rate = (
    ranked_2025["shortage_target"].mean()
)

cutoffs = [
    100,
    500,
    1000,
    2500,
    5000
]

capture_results = []

for k in cutoffs:

    top_k = ranked_2025.head(k)

    captured = int(
        top_k["shortage_target"].sum()
    )

    precision = (
        captured / k
    )

    recall = (
        captured
        / total_actual_shortages
        if total_actual_shortages > 0
        else 0
    )

    lift = (
        precision / baseline_rate
        if baseline_rate > 0
        else 0
    )

    capture_results.append(
        {
            "top_packages": k,
            "actual_shortages_captured": captured,
            "total_actual_shortages": total_actual_shortages,
            "shortage_capture_pct": recall * 100,
            "precision_pct": precision * 100,
            "lift_vs_random": lift
        }
    )


capture_table = pd.DataFrame(
    capture_results
)

print("PHARMAOPT — 2025 RISK-RANKING PERFORMANCE")
print("=" * 70)

capture_table

PHARMAOPT — 2025 RISK-RANKING PERFORMANCE


,top_packages,actual_shortages_captured,total_actual_shortages,shortage_capture_pct,precision_pct,lift_vs_random
0,100,0,96,0.000000,0.00,0.000000
1,500,3,96,3.125000,0.60,13.854188
2,1000,3,96,3.125000,0.30,6.927094
3,2500,14,96,14.583333,0.56,12.930575
4,5000,20,96,20.833333,0.40,9.236125


In [27]:
# ============================================================
# TOP 25 PHARMAOPT RISK FLAGS
# ============================================================

top_risk_2025 = ranked_2025[
    [
        "package_ndc",
        "generic_name",
        "brand_name",
        "labeler_name",
        "risk_score",
        "shortage_target"
    ]
].head(25)

top_risk_2025

,package_ndc,generic_name,brand_name,labeler_name,risk_score,shortage_target
0,0009-0870-26,clindamycin phosphate,Cleocin Phosphate,Pharmacia & Upjohn Company LLC,0.998976,0
1,0009-0775-26,clindamycin phosphate,Cleocin Phosphate,Pharmacia & Upjohn Company LLC,0.998976,0
2,0009-0902-18,clindamycin phosphate,Cleocin Phosphate,Pharmacia & Upjohn Company LLC,0.998976,0
3,0338-0013-29,Water,Sterile Water,Baxter Healthcare Company,0.998388,0
4,0338-0013-06,Water,Sterile Water,Baxter Healthcare Company,0.998388,0
5,0338-0013-08,Water,Sterile Water,Baxter Healthcare Company,0.998388,0
6,0270-0556-15,sincalide,KINEVAC,Bracco Diagnostics Inc.,0.998075,0
7,0264-7802-10,SODIUM CHLORIDE,Sodium Chloride,B. Braun Medical Inc.,0.998069,0
8,0264-7802-00,SODIUM CHLORIDE,Sodium Chloride,B. Braun Medical Inc.,0.998069,0
9,0264-7800-00,SODIUM CHLORIDE,Sodium Chloride,B. Braun Medical Inc.,0.998069,0


In [28]:
# ============================================================
# PHARMAOPT v0.1 — FINAL BASELINE PERFORMANCE AUDIT
# ============================================================

ranked_2025 = (
    results_2025
    .sort_values("risk_score", ascending=False)
    .reset_index(drop=True)
)

# Rank starts at 1 instead of 0
ranked_2025["risk_rank"] = (
    ranked_2025.index + 1
)

total_packages = len(ranked_2025)
total_shortages = int(
    ranked_2025["shortage_target"].sum()
)

baseline_rate = (
    total_shortages / total_packages
)

print("PHARMAOPT v0.1 — BASELINE PERFORMANCE")
print("=" * 75)

print("2025 packages:", f"{total_packages:,}")
print("Actual 2025 shortages:", f"{total_shortages:,}")
print(
    "Random baseline shortage rate:",
    f"{baseline_rate * 100:.4f}%"
)

print()
print("TOP-RISK CAPTURE")
print("-" * 75)

cutoffs = [
    100,
    500,
    1000,
    2500,
    5000,
    10000,
    25000
]

for k in cutoffs:

    top = ranked_2025.head(k)

    captured = int(
        top["shortage_target"].sum()
    )

    capture_pct = (
        captured / total_shortages * 100
    )

    precision = (
        captured / k
    )

    lift = (
        precision / baseline_rate
        if baseline_rate > 0
        else 0
    )

    print(
        f"Top {k:>6,}: "
        f"{captured:>3}/{total_shortages} shortages | "
        f"capture={capture_pct:>6.2f}% | "
        f"lift={lift:>6.2f}x"
    )


# ------------------------------------------------------------
# WHERE DID THE REAL SHORTAGES RANK?
# ------------------------------------------------------------

actual_shortages = (
    ranked_2025[
        ranked_2025["shortage_target"] == 1
    ]
    .copy()
)

print()
print("ACTUAL SHORTAGE RANK DISTRIBUTION")
print("-" * 75)

print(
    "Best-ranked actual shortage:",
    f"{actual_shortages['risk_rank'].min():,}"
)

print(
    "Median actual-shortage rank:",
    f"{actual_shortages['risk_rank'].median():,.0f}"
)

print(
    "Worst-ranked actual shortage:",
    f"{actual_shortages['risk_rank'].max():,}"
)

print()
print("TOP 10 HIGHEST-RANKED ACTUAL SHORTAGES")
print("-" * 75)

display(
    actual_shortages[
        [
            "risk_rank",
            "package_ndc",
            "generic_name",
            "brand_name",
            "labeler_name",
            "risk_score"
        ]
    ].head(10)
)

PHARMAOPT v0.1 — BASELINE PERFORMANCE
2025 packages: 221,667
Actual 2025 shortages: 96
Random baseline shortage rate: 0.0433%

TOP-RISK CAPTURE
---------------------------------------------------------------------------
Top    100:   0/96 shortages | capture=  0.00% | lift=  0.00x
Top    500:   3/96 shortages | capture=  3.12% | lift= 13.85x
Top  1,000:   3/96 shortages | capture=  3.12% | lift=  6.93x
Top  2,500:  14/96 shortages | capture= 14.58% | lift= 12.93x
Top  5,000:  20/96 shortages | capture= 20.83% | lift=  9.24x
Top 10,000:  24/96 shortages | capture= 25.00% | lift=  5.54x
Top 25,000:  46/96 shortages | capture= 47.92% | lift=  4.25x

ACTUAL SHORTAGE RANK DISTRIBUTION
---------------------------------------------------------------------------
Best-ranked actual shortage: 334
Median actual-shortage rank: 38,752
Worst-ranked actual shortage: 221,612

TOP 10 HIGHEST-RANKED ACTUAL SHORTAGES
---------------------------------------------------------------------------


,risk_rank,package_ndc,generic_name,brand_name,labeler_name,risk_score
333,334,0409-7101-67,Sodium Chloride,Sodium Chloride,"Hospira, Inc.",0.989918
334,335,0409-7101-02,Sodium Chloride,Sodium Chloride,"Hospira, Inc.",0.989918
431,432,0409-6637-14,Sodium Bicarbonate,Sodium Bicarbonate,"Hospira, Inc.",0.988185
1284,1285,0409-1176-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",0.968907
1285,1286,0409-1178-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",0.968859
1301,1302,0409-1181-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",0.968445
1648,1649,0049-3190-01,Voriconazole,VFEND,Roerig,0.960069
1729,1730,0641-6052-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,0.958968
1730,1731,0641-6054-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,0.958968
1731,1732,0641-6053-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,0.958968


In [29]:
# ============================================================
# PHARMAOPT v0.2
# HISTORICAL SHORTAGE FEATURE ENGINEERING
# ============================================================

enhanced_frames = []

print("BUILDING HISTORICAL SHORTAGE FEATURES")
print("=" * 70)


for year in [2022, 2023, 2024, 2025]:

    print(f"\nProcessing historical features for {year}...")

    index_date = pd.Timestamp(f"{year}-01-01")

    # Get this year's existing modeling rows
    year_df = pharmaopt_model_df[
        pharmaopt_model_df["prediction_year"] == year
    ].copy()


    # --------------------------------------------------------
    # HISTORICAL SHORTAGES BEFORE PREDICTION DATE
    # --------------------------------------------------------

    prior_shortages = query_df("""
    SELECT

        s.linked_package_ndc AS package_ndc,

        dp.product_ndc,
        dp.generic_name,
        dp.labeler_name,

        s.initial_posting_date

    FROM shortage_event s

    INNER JOIN drug_package pkg
        ON s.linked_package_ndc = pkg.package_ndc

    INNER JOIN drug_product dp
        ON pkg.product_id = dp.product_id

    WHERE s.linked_package_ndc IS NOT NULL
      AND s.initial_posting_date IS NOT NULL
      AND s.initial_posting_date < %s;
    """, (index_date.date(),))


    # --------------------------------------------------------
    # PACKAGE-LEVEL PRIOR SHORTAGES
    # --------------------------------------------------------

    package_history = (
        prior_shortages
        .groupby("package_ndc")
        .size()
        .to_dict()
    )

    year_df["package_prior_shortages"] = (
        year_df["package_ndc"]
        .map(package_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # PRODUCT-NDC PRIOR SHORTAGES
    # --------------------------------------------------------

    product_history = (
        prior_shortages
        .groupby("product_ndc")
        .size()
        .to_dict()
    )

    year_df["product_prior_shortages"] = (
        year_df["product_ndc"]
        .map(product_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # GENERIC-DRUG PRIOR SHORTAGES
    # --------------------------------------------------------

    generic_history = (
        prior_shortages
        .groupby("generic_name")
        .size()
        .to_dict()
    )

    year_df["generic_prior_shortages"] = (
        year_df["generic_name"]
        .map(generic_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # LABELER / MANUFACTURER PRIOR SHORTAGES
    # --------------------------------------------------------

    labeler_history = (
        prior_shortages
        .groupby("labeler_name")
        .size()
        .to_dict()
    )

    year_df["labeler_prior_shortages"] = (
        year_df["labeler_name"]
        .map(labeler_history)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # RECENT SHORTAGES — PRIOR 365 DAYS
    # --------------------------------------------------------

    one_year_before = (
        index_date - pd.DateOffset(years=1)
    )

    recent_shortages = prior_shortages[
        pd.to_datetime(
            prior_shortages["initial_posting_date"]
        ) >= one_year_before
    ].copy()


    generic_recent = (
        recent_shortages
        .groupby("generic_name")
        .size()
        .to_dict()
    )

    labeler_recent = (
        recent_shortages
        .groupby("labeler_name")
        .size()
        .to_dict()
    )


    year_df["generic_shortages_prior_1y"] = (
        year_df["generic_name"]
        .map(generic_recent)
        .fillna(0)
        .astype(int)
    )


    year_df["labeler_shortages_prior_1y"] = (
        year_df["labeler_name"]
        .map(labeler_recent)
        .fillna(0)
        .astype(int)
    )


    # --------------------------------------------------------
    # SIMPLE BINARY HISTORY FLAGS
    # --------------------------------------------------------

    year_df["package_has_shortage_history"] = (
        year_df["package_prior_shortages"] > 0
    ).astype(int)

    year_df["product_has_shortage_history"] = (
        year_df["product_prior_shortages"] > 0
    ).astype(int)

    year_df["generic_has_shortage_history"] = (
        year_df["generic_prior_shortages"] > 0
    ).astype(int)

    year_df["labeler_has_shortage_history"] = (
        year_df["labeler_prior_shortages"] > 0
    ).astype(int)


    enhanced_frames.append(year_df)


# ============================================================
# COMBINE ENHANCED PANEL
# ============================================================

pharmaopt_v02_df = pd.concat(
    enhanced_frames,
    ignore_index=True
)


print("\n" + "=" * 70)
print("PHARMAOPT v0.2 FEATURE TABLE COMPLETE")
print("=" * 70)

print(
    "Observations:",
    f"{len(pharmaopt_v02_df):,}"
)

print(
    "Features added:",
    10
)

print()

print(
    pharmaopt_v02_df[
        [
            "package_ndc",
            "prediction_year",
            "package_prior_shortages",
            "product_prior_shortages",
            "generic_prior_shortages",
            "labeler_prior_shortages",
            "shortage_target"
        ]
    ].head(10)
)

BUILDING HISTORICAL SHORTAGE FEATURES

Processing historical features for 2022...

Processing historical features for 2023...

Processing historical features for 2024...

Processing historical features for 2025...

PHARMAOPT v0.2 FEATURE TABLE COMPLETE
Observations: 762,907
Features added: 10

    package_ndc  prediction_year  package_prior_shortages  \
0  0002-0013-03             2022                        0   
1  0002-0013-10             2022                        0   
2  0002-0095-00             2022                        0   
3  0002-0096-00             2022                        0   
4  0002-0113-03             2022                        0   
5  0002-0113-10             2022                        0   
6  0002-0119-00             2022                        0   
7  0002-0121-00             2022                        0   
8  0002-0124-00             2022                        0   
9  0002-0124-03             2022                        0   

   product_prior_shortages  gener

In [30]:
# ============================================================
# PHARMAOPT v0.2 — SHORTAGE HISTORY MODEL
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


numeric_features_v02 = [

    # Existing features
    "product_age_days",

    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",

    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",

    "days_since_last_recall",
    "has_prior_recall",

    # NEW SUPPLY-HISTORY FEATURES
    "package_prior_shortages",
    "product_prior_shortages",
    "generic_prior_shortages",
    "labeler_prior_shortages",

    "generic_shortages_prior_1y",
    "labeler_shortages_prior_1y",

    "package_has_shortage_history",
    "product_has_shortage_history",
    "generic_has_shortage_history",
    "labeler_has_shortage_history"
]


categorical_features_v02 = [
    "dosage_form",
    "marketing_category"
]


features_v02 = (
    numeric_features_v02
    + categorical_features_v02
)


# ------------------------------------------------------------
# TIME SPLIT
# ------------------------------------------------------------

train_v02 = pharmaopt_v02_df[
    pharmaopt_v02_df["prediction_year"] < 2025
].copy()

test_v02 = pharmaopt_v02_df[
    pharmaopt_v02_df["prediction_year"] == 2025
].copy()


X_train_v02 = train_v02[features_v02]
y_train_v02 = train_v02["shortage_target"]

X_test_v02 = test_v02[features_v02]
y_test_v02 = test_v02["shortage_target"]


# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

numeric_pipe_v02 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipe_v02 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor_v02 = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe_v02,
            numeric_features_v02
        ),
        (
            "categorical",
            categorical_pipe_v02,
            categorical_features_v02
        )
    ]
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

pharmaopt_v02 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_v02
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1500,
                solver="liblinear"
            )
        )
    ]
)


print("Training PharmaOpt v0.2...")

pharmaopt_v02.fit(
    X_train_v02,
    y_train_v02
)

print("Training complete.")


# ------------------------------------------------------------
# RISK SCORES
# ------------------------------------------------------------

risk_v02 = pharmaopt_v02.predict_proba(
    X_test_v02
)[:, 1]


results_v02 = test_v02[
    [
        "package_ndc",
        "product_ndc",
        "generic_name",
        "brand_name",
        "labeler_name",
        "shortage_target"
    ]
].copy()

results_v02["risk_score"] = risk_v02


# ------------------------------------------------------------
# PERFORMANCE
# ------------------------------------------------------------

pr_auc_v02 = average_precision_score(
    y_test_v02,
    risk_v02
)

roc_auc_v02 = roc_auc_score(
    y_test_v02,
    risk_v02
)

base_rate_v02 = y_test_v02.mean()


print("\nPHARMAOPT v0.2 RESULTS")
print("=" * 65)

print(
    "PR-AUC:",
    f"{pr_auc_v02:.6f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_v02:.4f}"
)

print(
    "PR-AUC lift over random:",
    f"{pr_auc_v02 / base_rate_v02:.2f}x"
)

Training PharmaOpt v0.2...
Training complete.

PHARMAOPT v0.2 RESULTS
PR-AUC: 0.003329
ROC-AUC: 0.7644
PR-AUC lift over random: 7.69x


In [31]:
# ============================================================
# PHARMAOPT v0.2 — RISK CAPTURE
# ============================================================

ranked_v02 = (
    results_v02
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)

total_shortages = int(
    ranked_v02["shortage_target"].sum()
)

baseline_rate = (
    ranked_v02["shortage_target"].mean()
)


cutoffs = [
    100,
    500,
    1000,
    2500,
    5000,
    10000,
    25000
]


print("PHARMAOPT v0.2 — 2025 RISK CAPTURE")
print("=" * 75)


for k in cutoffs:

    top = ranked_v02.head(k)

    captured = int(
        top["shortage_target"].sum()
    )

    capture = (
        captured
        / total_shortages
        * 100
    )

    precision = (
        captured / k
    )

    lift = (
        precision
        / baseline_rate
    )

    print(
        f"Top {k:>6,}: "
        f"{captured:>3}/{total_shortages} | "
        f"capture={capture:>6.2f}% | "
        f"lift={lift:>7.2f}x"
    )

PHARMAOPT v0.2 — 2025 RISK CAPTURE
Top    100:   0/96 | capture=  0.00% | lift=   0.00x
Top    500:   5/96 | capture=  5.21% | lift=  23.09x
Top  1,000:   6/96 | capture=  6.25% | lift=  13.85x
Top  2,500:  11/96 | capture= 11.46% | lift=  10.16x
Top  5,000:  15/96 | capture= 15.62% | lift=   6.93x
Top 10,000:  40/96 | capture= 41.67% | lift=   9.24x
Top 25,000:  47/96 | capture= 48.96% | lift=   4.34x


In [32]:
# ============================================================
# PHARMAOPT v0.3
# SUPPLY-PRESSURE + RECENCY FEATURES
# ============================================================

import numpy as np

v03_frames = []

print("BUILDING PHARMAOPT v0.3 FEATURES")
print("=" * 75)


for year in [2022, 2023, 2024, 2025]:

    print(f"\nProcessing {year}...")

    index_date = pd.Timestamp(f"{year}-01-01")

    year_df = pharmaopt_v02_df[
        pharmaopt_v02_df["prediction_year"] == year
    ].copy()


    # ========================================================
    # 1. CURRENT ACTIVE PRODUCT/MARKET BREADTH
    # ========================================================

    product_breadth = (
        year_df.groupby("product_ndc")["package_ndc"]
        .nunique()
        .to_dict()
    )

    generic_breadth = (
        year_df.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_breadth = (
        year_df.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["product_active_packages"] = (
        year_df["product_ndc"]
        .map(product_breadth)
        .fillna(1)
    )

    year_df["generic_active_packages"] = (
        year_df["generic_name"]
        .map(generic_breadth)
        .fillna(1)
    )

    year_df["labeler_active_packages"] = (
        year_df["labeler_name"]
        .map(labeler_breadth)
        .fillna(1)
    )


    # ========================================================
    # 2. PRIOR SHORTAGE EVENTS
    # ========================================================

    history = query_df("""
        SELECT
            s.linked_package_ndc AS package_ndc,
            dp.product_ndc,
            dp.generic_name,
            dp.labeler_name,
            s.initial_posting_date

        FROM shortage_event s

        INNER JOIN drug_package pkg
            ON s.linked_package_ndc = pkg.package_ndc

        INNER JOIN drug_product dp
            ON pkg.product_id = dp.product_id

        WHERE s.linked_package_ndc IS NOT NULL
          AND s.initial_posting_date IS NOT NULL
          AND s.initial_posting_date < %s;
    """, (index_date.date(),))


    history["initial_posting_date"] = pd.to_datetime(
        history["initial_posting_date"]
    )


    # ========================================================
    # 3. LAST SHORTAGE DATE / RECENCY
    # ========================================================

    package_last = (
        history.groupby("package_ndc")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    product_last = (
        history.groupby("product_ndc")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    generic_last = (
        history.groupby("generic_name")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )

    labeler_last = (
        history.groupby("labeler_name")
        ["initial_posting_date"]
        .max()
        .to_dict()
    )


    year_df["package_last_shortage"] = (
        year_df["package_ndc"].map(package_last)
    )

    year_df["product_last_shortage"] = (
        year_df["product_ndc"].map(product_last)
    )

    year_df["generic_last_shortage"] = (
        year_df["generic_name"].map(generic_last)
    )

    year_df["labeler_last_shortage"] = (
        year_df["labeler_name"].map(labeler_last)
    )


    for level in [
        "package",
        "product",
        "generic",
        "labeler"
    ]:

        year_df[f"{level}_days_since_shortage"] = (
            index_date
            -
            pd.to_datetime(
                year_df[f"{level}_last_shortage"]
            )
        ).dt.days


    # ========================================================
    # 4. DISTINCT PACKAGES EVER AFFECTED
    # ========================================================

    product_affected = (
        history.groupby("product_ndc")["package_ndc"]
        .nunique()
        .to_dict()
    )

    generic_affected = (
        history.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_affected = (
        history.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["product_prior_affected_packages"] = (
        year_df["product_ndc"]
        .map(product_affected)
        .fillna(0)
    )

    year_df["generic_prior_affected_packages"] = (
        year_df["generic_name"]
        .map(generic_affected)
        .fillna(0)
    )

    year_df["labeler_prior_affected_packages"] = (
        year_df["labeler_name"]
        .map(labeler_affected)
        .fillna(0)
    )


    # ========================================================
    # 5. RECENT SHORTAGE HISTORY — PRIOR 365 DAYS
    # ========================================================

    one_year_before = (
        index_date - pd.DateOffset(years=1)
    )

    recent = history[
        history["initial_posting_date"] >= one_year_before
    ].copy()


    package_recent = (
        recent.groupby("package_ndc")
        .size()
        .to_dict()
    )

    product_recent = (
        recent.groupby("product_ndc")
        .size()
        .to_dict()
    )

    generic_recent_affected = (
        recent.groupby("generic_name")["package_ndc"]
        .nunique()
        .to_dict()
    )

    labeler_recent_affected = (
        recent.groupby("labeler_name")["package_ndc"]
        .nunique()
        .to_dict()
    )


    year_df["package_shortages_prior_1y"] = (
        year_df["package_ndc"]
        .map(package_recent)
        .fillna(0)
    )

    year_df["product_shortages_prior_1y"] = (
        year_df["product_ndc"]
        .map(product_recent)
        .fillna(0)
    )

    year_df["generic_recent_affected_packages"] = (
        year_df["generic_name"]
        .map(generic_recent_affected)
        .fillna(0)
    )

    year_df["labeler_recent_affected_packages"] = (
        year_df["labeler_name"]
        .map(labeler_recent_affected)
        .fillna(0)
    )


    # ========================================================
    # 6. NORMALIZED SUPPLY-PRESSURE FEATURES
    # ========================================================

    year_df["product_shortage_pressure"] = (
        year_df["product_prior_affected_packages"]
        /
        year_df["product_active_packages"]
    )

    year_df["generic_shortage_pressure"] = (
        year_df["generic_prior_affected_packages"]
        /
        year_df["generic_active_packages"]
    )

    year_df["labeler_shortage_pressure"] = (
        year_df["labeler_prior_affected_packages"]
        /
        year_df["labeler_active_packages"]
    )


    # ========================================================
    # 7. SIBLING PACKAGE SIGNAL
    # ========================================================

    year_df["product_sibling_shortage_packages"] = (
        year_df["product_prior_affected_packages"]
        -
        year_df["package_has_shortage_history"]
    ).clip(lower=0)


    year_df["generic_sibling_shortage_packages"] = (
        year_df["generic_prior_affected_packages"]
        -
        year_df["package_has_shortage_history"]
    ).clip(lower=0)


    # ========================================================
    # 8. LOG-TRANSFORM LARGE COUNTS
    # ========================================================

    year_df["log_labeler_prior_shortages"] = np.log1p(
        year_df["labeler_prior_shortages"]
    )

    year_df["log_generic_prior_shortages"] = np.log1p(
        year_df["generic_prior_shortages"]
    )


    v03_frames.append(year_df)


# ============================================================
# COMBINE
# ============================================================

pharmaopt_v03_df = pd.concat(
    v03_frames,
    ignore_index=True
)


print("\n" + "=" * 75)
print("PHARMAOPT v0.3 FEATURE TABLE COMPLETE")
print("=" * 75)

print(
    "Observations:",
    f"{len(pharmaopt_v03_df):,}"
)

print(
    "Shortage positives:",
    f"{pharmaopt_v03_df['shortage_target'].sum():,}"
)

BUILDING PHARMAOPT v0.3 FEATURES

Processing 2022...

Processing 2023...

Processing 2024...

Processing 2025...

PHARMAOPT v0.3 FEATURE TABLE COMPLETE
Observations: 762,907
Shortage positives: 426


In [33]:
# ============================================================
# PHARMAOPT v0.3 — SUPPLY PRESSURE MODEL
# ============================================================

features_v03_numeric = [

    # Product characteristics
    "product_age_days",

    # Recall history
    "prior_recall_count",
    "prior_class_I_count",
    "prior_class_II_count",
    "prior_class_III_count",
    "recalls_prior_1y",
    "recalls_prior_2y",
    "recalls_prior_5y",
    "days_since_last_recall",
    "has_prior_recall",

    # v0.2 shortage history
    "package_prior_shortages",
    "product_prior_shortages",
    "generic_prior_shortages",
    "labeler_prior_shortages",

    "generic_shortages_prior_1y",
    "labeler_shortages_prior_1y",

    "package_has_shortage_history",
    "product_has_shortage_history",
    "generic_has_shortage_history",
    "labeler_has_shortage_history",

    # NEW v0.3 breadth
    "product_active_packages",
    "generic_active_packages",
    "labeler_active_packages",

    # NEW recency
    "package_days_since_shortage",
    "product_days_since_shortage",
    "generic_days_since_shortage",
    "labeler_days_since_shortage",

    # NEW affected-package breadth
    "product_prior_affected_packages",
    "generic_prior_affected_packages",
    "labeler_prior_affected_packages",

    # NEW recent pressure
    "package_shortages_prior_1y",
    "product_shortages_prior_1y",
    "generic_recent_affected_packages",
    "labeler_recent_affected_packages",

    # NEW normalized pressure
    "product_shortage_pressure",
    "generic_shortage_pressure",
    "labeler_shortage_pressure",

    # NEW sibling signal
    "product_sibling_shortage_packages",
    "generic_sibling_shortage_packages",

    # Log counts
    "log_labeler_prior_shortages",
    "log_generic_prior_shortages"
]


features_v03_cat = [
    "dosage_form",
    "marketing_category"
]


features_v03 = (
    features_v03_numeric
    +
    features_v03_cat
)


train_v03 = pharmaopt_v03_df[
    pharmaopt_v03_df["prediction_year"] < 2025
].copy()

test_v03 = pharmaopt_v03_df[
    pharmaopt_v03_df["prediction_year"] == 2025
].copy()


X_train_v03 = train_v03[features_v03]
y_train_v03 = train_v03["shortage_target"]

X_test_v03 = test_v03[features_v03]
y_test_v03 = test_v03["shortage_target"]


numeric_pipe_v03 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipe_v03 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor_v03 = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe_v03,
            features_v03_numeric
        ),
        (
            "categorical",
            categorical_pipe_v03,
            features_v03_cat
        )
    ]
)


pharmaopt_v03 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_v03
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                solver="liblinear"
            )
        )
    ]
)


print("Training PharmaOpt v0.3...")

pharmaopt_v03.fit(
    X_train_v03,
    y_train_v03
)

print("Training complete.")


risk_v03 = pharmaopt_v03.predict_proba(
    X_test_v03
)[:, 1]


pr_auc_v03 = average_precision_score(
    y_test_v03,
    risk_v03
)

roc_auc_v03 = roc_auc_score(
    y_test_v03,
    risk_v03
)


results_v03 = test_v03[
    [
        "package_ndc",
        "product_ndc",
        "generic_name",
        "brand_name",
        "labeler_name",
        "shortage_target"
    ]
].copy()

results_v03["risk_score"] = risk_v03


ranked_v03 = (
    results_v03
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)


total_shortages = int(
    ranked_v03["shortage_target"].sum()
)

baseline_rate = (
    ranked_v03["shortage_target"].mean()
)


print("\nPHARMAOPT v0.3")
print("=" * 75)

print(
    "PR-AUC:",
    f"{pr_auc_v03:.6f}"
)

print(
    "PR-AUC lift:",
    f"{pr_auc_v03 / baseline_rate:.2f}x"
)

print(
    "ROC-AUC:",
    f"{roc_auc_v03:.4f}"
)

print()

for k in [
    100,
    500,
    1000,
    2500,
    5000,
    10000,
    25000
]:

    top = ranked_v03.head(k)

    captured = int(
        top["shortage_target"].sum()
    )

    capture_pct = (
        captured /
        total_shortages *
        100
    )

    lift = (
        (captured / k)
        /
        baseline_rate
    )

    print(
        f"Top {k:>6,}: "
        f"{captured:>3}/{total_shortages} | "
        f"capture={capture_pct:>6.2f}% | "
        f"lift={lift:>7.2f}x"
    )
    

Training PharmaOpt v0.3...
Training complete.

PHARMAOPT v0.3
PR-AUC: 0.003345
PR-AUC lift: 7.72x
ROC-AUC: 0.7988

Top    100:   0/96 | capture=  0.00% | lift=   0.00x
Top    500:   7/96 | capture=  7.29% | lift=  32.33x
Top  1,000:   8/96 | capture=  8.33% | lift=  18.47x
Top  2,500:  13/96 | capture= 13.54% | lift=  12.01x
Top  5,000:  19/96 | capture= 19.79% | lift=   8.77x
Top 10,000:  30/96 | capture= 31.25% | lift=   6.93x
Top 25,000:  45/96 | capture= 46.88% | lift=   4.16x


In [37]:
# ============================================================
# PHARMAOPT — CMS PART D UTILIZATION INGESTION
# NOTEBOOK 07
# ============================================================

import pandas as pd
import os
import glob

print("Current folder:")
print(os.getcwd())

print("\nCSV files found:")
cms_files = glob.glob("*.csv")

for file in cms_files:
    print(" -", file)

Current folder:
C:\Users\jpiza\PharmaOpt

CSV files found:


In [39]:
# ============================================================
# FIND CMS FILES
# ============================================================

from pathlib import Path
import os

current_folder = Path.cwd()

print("NOTEBOOK IS RUNNING FROM:")
print(current_folder)

print("\nFILES IN THIS FOLDER:")
print("=" * 70)

all_files = list(current_folder.iterdir())

for file in all_files:
    if file.is_file():
        print(file.name)

print("\nPOSSIBLE CMS DATA FILES:")
print("=" * 70)

data_extensions = {
    ".csv",
    ".zip",
    ".xlsx",
    ".parquet"
}

possible_data_files = [
    file
    for file in all_files
    if file.is_file()
    and file.suffix.lower() in data_extensions
]

if possible_data_files:
    for file in possible_data_files:
        print(
            file.name,
            " | type:",
            file.suffix.lower()
        )
else:
    print("NO CSV, ZIP, XLSX, OR PARQUET FILES FOUND.")

NOTEBOOK IS RUNNING FROM:
C:\Users\jpiza\PharmaOpt

FILES IN THIS FOLDER:
01_fda_shortage_audit.ipynb.ipynb
02_fda_ndc_audit.ipynb.ipynb
03_fda_recall_audit.ipynb.ipynb
04_fda_data_integration.ipynb.ipynb
05_mysql_data_load.ipynb.ipynb
06_pharmaopt_analytics.ipynb.ipynb
drug-enforcement-0001-of-0001.json.zip
drug-ndc-0001-of-0001.json.zip
drug-shortages-0001-of-0001.json.zip

POSSIBLE CMS DATA FILES:
drug-enforcement-0001-of-0001.json.zip  | type: .zip
drug-ndc-0001-of-0001.json.zip  | type: .zip
drug-shortages-0001-of-0001.json.zip  | type: .zip


In [40]:
# ============================================================
# PHARMAOPT v0.4
# IMPORT CMS DEMAND FEATURES
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

PROJECT_DIR = Path.cwd()

cms_features = pd.read_csv(
    PROJECT_DIR / "cms_generic_demand_features.csv"
)

print("CMS feature rows:", f"{len(cms_features):,}")
print()

print("CMS prediction years:")
print(
    cms_features["prediction_year"]
    .value_counts()
    .sort_index()
)

display(cms_features.head())

CMS feature rows: 7,737

CMS prediction years:
prediction_year
2022    1884
2023    1920
2024    1945
2025    1988
Name: count, dtype: int64


,generic_normalized,cms_data_year,prediction_year,cms_total_claims,cms_30day_fills,cms_total_drug_cost,cms_brand_count,cms_claim_growth,cms_fill_growth,log_cms_claims,log_cms_fills,log_cms_cost,fda_generic_match
0,0 9 sodium chloride,2021,2022,248597,251845.1,5432729.48,1,NaN,NaN,12.423592,12.436573,15.507952,True
1,0 9 sodium chloride,2022,2023,259126,262867.4,6188339.70,1,0.042354,0.043766,12.465074,12.479409,15.638178,True
2,0 9 sodium chloride,2023,2024,303702,310872.7,7568584.50,1,0.172024,0.182622,12.623806,12.647142,15.839517,True
3,0 9 sodium chloride,2024,2025,345300,351967.5,8234898.64,1,0.136970,0.132192,12.752172,12.771297,15.923892,True
4,aa 4 25 calcium lytes dex 10,2022,2023,12,12.0,3636.26,1,NaN,NaN,2.564949,2.564949,8.198986,False


In [41]:
# ============================================================
# LINK CMS DEMAND → PHARMAOPT PACKAGE-YEAR PANEL
# ============================================================

def normalize_drug_name(value):

    if pd.isna(value):
        return None

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = value.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


pharmaopt_v04_df = pharmaopt_v03_df.copy()

pharmaopt_v04_df["generic_normalized"] = (
    pharmaopt_v04_df["generic_name"]
    .apply(normalize_drug_name)
)


cms_merge_columns = [
    "generic_normalized",
    "prediction_year",
    "cms_total_claims",
    "cms_30day_fills",
    "cms_total_drug_cost",
    "cms_brand_count",
    "cms_claim_growth",
    "cms_fill_growth",
    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost"
]


pharmaopt_v04_df = pharmaopt_v04_df.merge(
    cms_features[cms_merge_columns],
    on=[
        "generic_normalized",
        "prediction_year"
    ],
    how="left",
    validate="many_to_one"
)


pharmaopt_v04_df["cms_demand_match"] = (
    pharmaopt_v04_df["cms_total_claims"]
    .notna()
    .astype(int)
)


print("PHARMAOPT v0.4 MERGE COMPLETE")
print("=" * 75)

print(
    "Package-year observations:",
    f"{len(pharmaopt_v04_df):,}"
)

print()

for year in [2022, 2023, 2024, 2025]:

    temp = pharmaopt_v04_df[
        pharmaopt_v04_df["prediction_year"] == year
    ]

    matched = int(
        temp["cms_demand_match"].sum()
    )

    total = len(temp)

    positives = temp[
        temp["shortage_target"] == 1
    ]

    positive_matches = int(
        positives["cms_demand_match"].sum()
    )

    print(
        f"{year}: "
        f"{matched:,}/{total:,} packages linked to CMS "
        f"({matched / total * 100:.2f}%) | "
        f"shortage cases linked: "
        f"{positive_matches}/{len(positives)}"
    )

PHARMAOPT v0.4 MERGE COMPLETE
Package-year observations: 762,907

2022: 67,161/157,804 packages linked to CMS (42.56%) | shortage cases linked: 5/128
2023: 72,868/179,808 packages linked to CMS (40.53%) | shortage cases linked: 105/195
2024: 79,643/203,628 packages linked to CMS (39.11%) | shortage cases linked: 1/7
2025: 86,361/221,667 packages linked to CMS (38.96%) | shortage cases linked: 72/96


In [42]:
# ============================================================
# PHARMAOPT v0.4
# SUPPLY PRESSURE + DEMAND PRESSURE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)


# Start with all v0.3 numeric features
features_v04_numeric = features_v03_numeric + [

    # CMS DEMAND LEVEL
    "log_cms_claims",
    "log_cms_fills",
    "log_cms_cost",

    # Product-market breadth within CMS
    "cms_brand_count",

    # Demand acceleration
    "cms_claim_growth",
    "cms_fill_growth",

    # Explicitly tell model whether CMS matched
    "cms_demand_match"
]


features_v04_cat = [
    "dosage_form",
    "marketing_category"
]


features_v04 = (
    features_v04_numeric
    +
    features_v04_cat
)


# ============================================================
# TEMPORAL SPLIT
# ============================================================

train_v04 = pharmaopt_v04_df[
    pharmaopt_v04_df["prediction_year"] < 2025
].copy()

test_v04 = pharmaopt_v04_df[
    pharmaopt_v04_df["prediction_year"] == 2025
].copy()


X_train_v04 = train_v04[features_v04]
y_train_v04 = train_v04["shortage_target"]

X_test_v04 = test_v04[features_v04]
y_test_v04 = test_v04["shortage_target"]


print("TRAIN")
print(
    f"{len(X_train_v04):,} observations | "
    f"{y_train_v04.sum():,} shortages"
)

print()

print("2025 DEVELOPMENT HOLDOUT")
print(
    f"{len(X_test_v04):,} observations | "
    f"{y_test_v04.sum():,} shortages"
)


# ============================================================
# PREPROCESSING
# ============================================================

numeric_pipe_v04 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_pipe_v04 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor_v04 = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe_v04,
            features_v04_numeric
        ),
        (
            "categorical",
            categorical_pipe_v04,
            features_v04_cat
        )
    ]
)


# ============================================================
# MODEL
# ============================================================

pharmaopt_v04 = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_v04
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2500,
                solver="liblinear"
            )
        )
    ]
)


print("\nTraining PharmaOpt v0.4...")

pharmaopt_v04.fit(
    X_train_v04,
    y_train_v04
)

print("Training complete.")


# ============================================================
# 2025 RISK RANKING
# ============================================================

risk_v04 = pharmaopt_v04.predict_proba(
    X_test_v04
)[:, 1]


results_v04 = test_v04[
    [
        "package_ndc",
        "product_ndc",
        "generic_name",
        "brand_name",
        "labeler_name",
        "shortage_target",
        "cms_demand_match",
        "cms_total_claims",
        "cms_30day_fills",
        "cms_claim_growth"
    ]
].copy()


results_v04["risk_score"] = risk_v04


ranked_v04 = (
    results_v04
    .sort_values(
        "risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# PERFORMANCE
# ============================================================

pr_auc_v04 = average_precision_score(
    y_test_v04,
    risk_v04
)

roc_auc_v04 = roc_auc_score(
    y_test_v04,
    risk_v04
)

baseline_rate = y_test_v04.mean()


print("\nPHARMAOPT v0.4")
print("=" * 75)

print(
    "PR-AUC:",
    f"{pr_auc_v04:.6f}"
)

print(
    "PR-AUC lift over random:",
    f"{pr_auc_v04 / baseline_rate:.2f}x"
)

print(
    "ROC-AUC:",
    f"{roc_auc_v04:.4f}"
)


print("\nRISK CAPTURE")
print("-" * 75)


total_shortages = int(
    ranked_v04["shortage_target"].sum()
)


for k in [
    100,
    500,
    1000,
    2500,
    5000,
    10000,
    25000
]:

    top = ranked_v04.head(k)

    captured = int(
        top["shortage_target"].sum()
    )

    capture_pct = (
        captured /
        total_shortages *
        100
    )

    lift = (
        (captured / k)
        /
        baseline_rate
    )

    print(
        f"Top {k:>6,}: "
        f"{captured:>3}/{total_shortages} | "
        f"capture={capture_pct:>6.2f}% | "
        f"lift={lift:>7.2f}x"
    )

TRAIN
541,240 observations | 330 shortages

2025 DEVELOPMENT HOLDOUT
221,667 observations | 96 shortages

Training PharmaOpt v0.4...
Training complete.

PHARMAOPT v0.4
PR-AUC: 0.003133
PR-AUC lift over random: 7.23x
ROC-AUC: 0.7769

RISK CAPTURE
---------------------------------------------------------------------------
Top    100:   0/96 | capture=  0.00% | lift=   0.00x
Top    500:  10/96 | capture= 10.42% | lift=  46.18x
Top  1,000:  10/96 | capture= 10.42% | lift=  23.09x
Top  2,500:  13/96 | capture= 13.54% | lift=  12.01x
Top  5,000:  17/96 | capture= 17.71% | lift=   7.85x
Top 10,000:  24/96 | capture= 25.00% | lift=   5.54x
Top 25,000:  47/96 | capture= 48.96% | lift=   4.34x


In [44]:
# ============================================================
# PHARMAOPT — MODEL EXPERIMENT COMPARISON
# v0.3 vs v0.4
# ============================================================

model_comparison = pd.DataFrame({

    "model": [
        "v0.3 FDA Supply",
        "v0.4 FDA + CMS Demand"
    ],

    "PR_AUC": [
        0.003345,
        0.003133
    ],

    "PR_AUC_Lift": [
        7.72,
        7.23
    ],

    "ROC_AUC": [
        0.7988,
        0.7769
    ],

    "Top_500_Captured": [
        7,
        10
    ],

    "Top_1000_Captured": [
        8,
        10
    ],

    "Top_2500_Captured": [
        13,
        13
    ],

    "Top_5000_Captured": [
        19,
        17
    ],

    "Top_10000_Captured": [
        30,
        24
    ],

    "Top_25000_Captured": [
        45,
        47
    ]
})

display(model_comparison)

,model,PR_AUC,PR_AUC_Lift,ROC_AUC,Top_500_Captured,Top_1000_Captured,Top_2500_Captured,Top_5000_Captured,Top_10000_Captured,Top_25000_Captured
0,v0.3 FDA Supply,0.003345,7.72,0.7988,7,8,13,19,30,45
1,v0.4 FDA + CMS Demand,0.003133,7.23,0.7769,10,10,13,17,24,47


In [45]:
# ============================================================
# PHARMAOPT — CMS COVERAGE DIAGNOSTIC
# ============================================================

print("CMS DEMAND COVERAGE")
print("=" * 75)

for year in sorted(
    pharmaopt_v04_df["prediction_year"].unique()
):

    temp = pharmaopt_v04_df[
        pharmaopt_v04_df["prediction_year"] == year
    ]

    matched = temp[
        temp["cms_demand_match"] == 1
    ]

    shortages = temp[
        temp["shortage_target"] == 1
    ]

    matched_shortages = shortages[
        shortages["cms_demand_match"] == 1
    ]

    print(f"\n{year}")
    print("-" * 40)

    print(
        "Package coverage:",
        f"{len(matched):,}/{len(temp):,}",
        f"({len(matched) / len(temp) * 100:.2f}%)"
    )

    print(
        "Shortage coverage:",
        f"{len(matched_shortages):,}/{len(shortages):,}",
        f"({len(matched_shortages) / len(shortages) * 100:.2f}%)"
        if len(shortages) > 0
        else "N/A"
    )

CMS DEMAND COVERAGE

2022
----------------------------------------
Package coverage: 67,161/157,804 (42.56%)
Shortage coverage: 5/128 (3.91%)

2023
----------------------------------------
Package coverage: 72,868/179,808 (40.53%)
Shortage coverage: 105/195 (53.85%)

2024
----------------------------------------
Package coverage: 79,643/203,628 (39.11%)
Shortage coverage: 1/7 (14.29%)

2025
----------------------------------------
Package coverage: 86,361/221,667 (38.96%)
Shortage coverage: 72/96 (75.00%)


In [46]:
# ============================================================
# PHARMAOPT — TOP v0.4 RISK SIGNALS
# ============================================================

top_v04 = ranked_v04.head(500).copy()

print("TOP 500 v0.4 RISK GROUP")
print("=" * 75)

print(
    "Actual 2025 shortages captured:",
    int(top_v04["shortage_target"].sum())
)

print(
    "CMS matched:",
    int(top_v04["cms_demand_match"].sum()),
    "/",
    len(top_v04)
)

print(
    "CMS match rate:",
    f"{top_v04['cms_demand_match'].mean() * 100:.2f}%"
)

print("\nACTUAL SHORTAGES FOUND IN TOP 500")
print("=" * 75)

display(
    top_v04[
        top_v04["shortage_target"] == 1
    ][
        [
            "package_ndc",
            "generic_name",
            "brand_name",
            "labeler_name",
            "cms_total_claims",
            "cms_30day_fills",
            "cms_claim_growth",
            "risk_score"
        ]
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
)

TOP 500 v0.4 RISK GROUP
Actual 2025 shortages captured: 10
CMS matched: 314 / 500
CMS match rate: 62.80%

ACTUAL SHORTAGES FOUND IN TOP 500


,package_ndc,generic_name,brand_name,labeler_name,cms_total_claims,cms_30day_fills,cms_claim_growth,risk_score
271,0409-6637-14,Sodium Bicarbonate,Sodium Bicarbonate,"Hospira, Inc.",8520.0,8731.7,-0.118651,0.999293
306,0409-1176-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",NaN,NaN,NaN,0.999135
307,0409-1178-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",NaN,NaN,NaN,0.999133
311,0409-1181-30,MEPERIDINE HYDROCHLORIDE,DEMEROL,"Hospira, Inc.",NaN,NaN,NaN,0.999117
407,0641-6052-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,NaN,NaN,NaN,0.998513
408,0641-6053-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,NaN,NaN,NaN,0.998513
409,0641-6054-25,MEPERIDINE HYDROCHLORIDE,MEPERIDINE HYDROCHLORIDE,Hikma Pharmaceuticals USA Inc.,NaN,NaN,NaN,0.998513
455,0409-7101-67,Sodium Chloride,Sodium Chloride,"Hospira, Inc.",615.0,615.8,0.196498,0.998332
456,0409-7101-02,Sodium Chloride,Sodium Chloride,"Hospira, Inc.",615.0,615.8,0.196498,0.998332
471,25021-230-02,irinotecan hydrochloride,Irinotecan Hydrochloride,Sagent Pharmaceuticals,NaN,NaN,NaN,0.998278


In [47]:
# ============================================================
# SAVE PHARMAOPT v0.4 RESULTS
# ============================================================

results_file = (
    PROJECT_DIR /
    "pharmaopt_v04_2025_risk_ranking.csv"
)

ranked_v04.to_csv(
    results_file,
    index=False
)

comparison_file = (
    PROJECT_DIR /
    "pharmaopt_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_file,
    index=False
)

print("SAVED:")
print(results_file)
print(comparison_file)

SAVED:
C:\Users\jpiza\PharmaOpt\pharmaopt_v04_2025_risk_ranking.csv
C:\Users\jpiza\PharmaOpt\pharmaopt_model_comparison.csv
